In [6]:
import numpy as np
import pandas as pd
import dowhy
from dowhy import CausalModel

# Set a random seed for reproducibility
np.random.seed(42)
num_samples = 5000

# Confounder (e.g., Socioeconomic status)
W = np.random.normal(loc=0, scale=1, size=num_samples)

# Treatment (e.g., Taking a training course) - influenced by the confounder W
treatment_prob = 1 / (1 + np.exp(-W))
X = np.random.binomial(1, treatment_prob)

# Outcome (e.g., Income) - Fixed 'mean' to 'loc' here as well!
Y = 2.0 * X + 3.0 * W + np.random.normal(loc=0, scale=1, size=num_samples)

# Combine into a DataFrame
df = pd.DataFrame({'W': W, 'Treatment': X, 'Outcome': Y})
print(df.head())

          W  Treatment   Outcome
0  0.496714          1  2.551675
1 -0.138264          0 -0.961201
2  0.647689          1  3.660825
3  1.523030          1  7.619864
4 -0.234153          0 -1.865399


In [7]:
# 1. Create a causal model and define the relationships
model = CausalModel(
    data=df,
    treatment='Treatment',
    outcome='Outcome',
    common_causes=['W'] # W is our confounder affecting both X and Y
)

# 2. Identify the causal effect expression
identified_estimand = model.identify_effect()
print(identified_estimand)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                    
────────────(E[Outcome|W])
d[Treatment]              
Estimand assumption 1, Unconfoundedness: If U→{Treatment} and U→Outcome then P(Outcome|Treatment,W,U) = P(Outcome|Treatment,W)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
     d                    
────────────(E[Outcome|W])
d[Treatment]              
Estimand assumption 1, Unconfoundedness: If U→{Treatment} and U→Outcome then P(Outcome|Treatment,W,U) = P(Outcome|Treatment,W)



In [8]:
# 3. Estimate the causal effect
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_stratification"
)

print(f"Estimated Causal Effect: {estimate.value}")
print(f"Actual True Causal Effect: 2.0")

Estimated Causal Effect: 2.036270360593987
Actual True Causal Effect: 2.0


In [9]:
# 4. Refute the estimate
refute_results = model.refute_estimate(
    identified_estimand, 
    estimate,
    method_name="random_common_cause"
)
print(refute_results)

Refute: Add a random common cause
Estimated effect:2.036270360593987
New effect:2.0362703605939876
p value:1.0

